# Loss-model comparison: parametric baseline vs. NTM

Addresses reviewers 1 & 3 (additional baselines / benchmarking). All models predict the **same** correction target — the residual `measured - analytical` (equivalently the loss torque `analytical - measured`) — on the **same** 0.15/seed-42 test split, so the RMSEs are directly comparable to the NTM's 0.034 Nm.

Parametric loss model (iron + mechanical losses, linear in 4 coefficients, ordinary least squares):

$$T_{loss}(\omega, i_s) = T_c + B\,\omega + k_h\,\lVert i_s\rVert^2 + k_e\,\omega\lVert i_s\rVert^2$$

with named contributions: $T_c$ constant (Coulomb friction + no-load hysteresis), $B\omega$ viscous friction + speed-dependent no-load iron loss, $k_h\lVert i_s\rVert^2$ load-dependent hysteresis iron loss, $k_e\,\omega\lVert i_s\rVert^2$ load-dependent eddy iron loss. $\lVert i_s\rVert$ is the full stator-current magnitude (3rd harmonic included).

In [ ]:
import os
import sys

import numpy as np
import torch

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.parameters import Flux_IEEEMachine2, IEEEMachine2
from current_setpoints.optimization import ModelAnalytical
from current_setpoints.utils import (
    IRON_LOSS_TERMS,
    build_residual_test_set,
    fit_parametric_loss,
    load_aggregated_csv_data,
    load_neural_model,
    loss_model_rmse,
)

DATA_DIR = os.path.join(root_dir, "data")
CSV_PATH = os.path.join(DATA_DIR, "aggregated_file_means.csv")
WEIGHTS = os.path.join(root_dir, "weights", "NTM_Weights_0344.pth")
SCALER = os.path.join(root_dir, "weights", "NTM_Scaler_0344.npy")
HIDDEN_SIZE, INPUT_SIZE = 12, 5
DEVICE = torch.device("cpu")

## Data: residual target on the 85% / 15% split

Fit the parametric model on the same 85% train-val portion the NTM was fit on; evaluate everything on the held-out 15%.

In [ ]:
COL_MAP = {c: c for c in ["omega", "id1", "iq1", "id3", "iq3", "torq"]}
df = load_aggregated_csv_data(CSV_PATH, COL_MAP)
X = df[["omega", "id1", "iq1", "id3", "iq3"]].values
y_meas = df[["torq"]].values

machine = IEEEMachine2()
machine.set_max_pars(curr_max=30.0, volt_max=13.0, omega_max=1800)
analytical = ModelAnalytical(machine=machine, flux=Flux_IEEEMachine2())

X_train, X_test, y_train, y_test = build_residual_test_set(
    X, y_meas, analytical, return_train=True
)

# Loss torque = analytical - measured = -(residual). Fitting this gives
# positive, physically interpretable coefficients; RMSE is identical to the
# residual RMSE (sign-symmetric), hence comparable to the NTM.
Tloss_train = -y_train.ravel()
Tloss_test = -y_test.ravel()
print(f"train points: {len(X_train)}   test points: {len(X_test)}")

## Fit the parametric loss model

Coefficients should come out physically positive ($B, k_h, k_e \ge 0$).

In [ ]:
m_param = fit_parametric_loss(X_train, Tloss_train, terms=IRON_LOSS_TERMS)

print("parametric loss coefficients (for the record; report R2/RMSE in the paper):")
for k, v in m_param["named"].items():
    print(f"  {k:5s} = {v: .6g}")
print(f"  train R2 = {m_param['r2']:.4f}")

## Comparison table (test-set RMSE)

In [ ]:
# Analytical only: predicts zero correction -> RMSE is the RMS of the residual.
rmse_analytical = float(np.sqrt(np.mean(y_test.ravel() ** 2)))
rmse_param = loss_model_rmse(m_param, X_test, Tloss_test)

# NTM: predicts the residual directly.
net, scaler = load_neural_model(
    WEIGHTS, SCALER, hidden_size=HIDDEN_SIZE, input_size=INPUT_SIZE, device=DEVICE
)
with torch.no_grad():
    ntm_pred = net(torch.from_numpy(scaler.transform(X_test)).float().to(DEVICE)).cpu().numpy().ravel()
rmse_ntm = float(np.sqrt(np.mean((ntm_pred - y_test.ravel()) ** 2)))

rows = [
    ("analytical only (no correction)", 0, rmse_analytical),
    ("parametric loss model", 4, rmse_param),
    ("NTM (SiLU, _0344)", 85, rmse_ntm),
]
print(f"{'model':<34} | {'#params':>7} | {'test RMSE [Nm]':>14}")
print("-" * 62)
for name, n, r in rows:
    print(f"{name:<34} | {n:>7} | {r:>14.4f}")

**Reading it.** The uncorrected analytical model leaves an RMSE of ~0.16 Nm. A parametric model of the iron and mechanical losses reduces this to ~0.09 Nm but cannot represent the harmonic-dependent component of the correction. The NTM, which resolves the full current vector, reaches 0.034 Nm — roughly a 3x improvement over the parametric baseline — which justifies the neural correction and lets it serve as the objective for 3rd-harmonic-injection optimization.